In [0]:
CATALOG = "dbs_main_catalog"
SCHEMA = "northwind_bronze"

# Auto Loader stores its schema versions here
schema_path = f"/Volumes/{CATALOG}/{SCHEMA}/_internal/_schemas/orders"

print("=== Schema location contents ===")
for f in dbutils.fs.ls(schema_path):
    print(f"  {f.path} | size={f.size} | isDir={f.isDir()}")

In [0]:
print("=== Schema versions ===")
for f in dbutils.fs.ls(f"{schema_path}/_schemas"):
    print(f"  {f.name} | {f.size:,} bytes")

In [0]:
# Read each schema version as text to see what Auto Loader inferred
for f in sorted(dbutils.fs.ls(f"{schema_path}/_schemas"), key=lambda x: x.name):
    print(f"\n========== Schema Version {f.name} ==========")
    content = dbutils.fs.head(f.path, max_bytes=10_000)
    print(content)

In [0]:
spark.sql(f"DESCRIBE HISTORY {CATALOG}.{SCHEMA}.orders_raw").show(truncate=False)

In [0]:
# How many customers have multiple distinct addresses across snapshots?
addr_changes = (customers_df.groupBy("customer_id")
                .agg(F.countDistinct(F.col("address.street")).alias("distinct_addrs"))
                .filter(F.col("distinct_addrs") > 1))
print(f"Customers with address changes: {addr_changes.count()}")

In [0]:
CATALOG = "dbs_main_catalog"
SCHEMA = "northwind_bronze"

# Remove existing customer landing files (we'll regenerate)
dbutils.fs.rm(f"/Volumes/{CATALOG}/{SCHEMA}/landing/customers", recurse=True)

# Drop the Bronze table and its checkpoint/schema
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.customers_raw")
dbutils.fs.rm(f"/Volumes/{CATALOG}/{SCHEMA}/_internal/_checkpoints/bronze_customers", recurse=True)
dbutils.fs.rm(f"/Volumes/{CATALOG}/{SCHEMA}/_internal/_schemas/customers", recurse=True)

print("✅ Customer state cleared")